# 02. 전처리 (Step 1)

01번 노트북에서 확인한 실측 함정을 반영하여 외국인(GENDER_CD='3') 소비 데이터를
정제하고 `data/processed/foreign_consumption_clean.csv`로 저장한다.

In [1]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
from data_loader import load_raw_data, validate_business_categories
from preprocessing import clean_and_filter_foreign, save_processed, PROCESSED_PATH, AGE_LABELS

pd.set_option('display.max_columns', 20)


## 2-1. 로딩 + 정제 + 외국인 필터링

`preprocessing.clean_and_filter_foreign()`은 업종명 정규화, 외국인 필터링, AGE_CD 한글 라벨 매핑, 건당 평균 결제금액 계산을 한 번에 수행한다.

In [2]:
raw = load_raw_data()
foreign_df = clean_and_filter_foreign(raw)
validate_business_categories(foreign_df)

print(f"외국인 데이터 행 수: {len(foreign_df):,} (원본 대비 {len(foreign_df)/len(raw)*100:.1f}%)")
foreign_df.head()


외국인 데이터 행 수: 70,178 (원본 대비 28.9%)


,STRD_YYMM,SIDO_NM,CCG_NM,GENDER_CD,AGE_CD,TP_BUZ_NO,TP_BUZ_NM,amt,cnt,AGE_LABEL,amt_per_cnt
0,202605,전북특별자치도,익산시,3,5,8001,일반한식,55830000,1385,50대,40310.469314
1,202601,경기도,수원시 팔달구,3,4,8006,서양음식,30810000,2289,40대,13460.026212
2,202604,대전광역시,대덕구,3,5,4020,슈퍼마켓,25330000,1266,50대,20007.898894
3,202606,경기도,안산시 단원구,3,2,4010,편의점,226020000,28029,20대,8063.791074
4,202603,서울특별시,강남구,3,1,4010,편의점,9360000,1350,20대 이하,6933.333333


## 2-2. 결측/0 이하 값 처리 방침

`preprocessing.clean_and_filter_foreign()` 내부에서 amt/cnt에 결측 또는 0 이하 값이
없음을 assert로 고정했다 (242,574행 전수 검사 결과 0건). 즉 원본 데이터는 이미
"집계값이 1건 이상 존재하는 조합만 수록"된 형태로 제공되므로, 별도의 결측치 대체나
행 제거 로직을 추가하지 않았다.

In [3]:
assert foreign_df['amt'].isnull().sum() == 0
assert foreign_df['cnt'].isnull().sum() == 0
assert (foreign_df['amt'] <= 0).sum() == 0
assert (foreign_df['cnt'] <= 0).sum() == 0
print("[확인] 외국인 데이터에 결측/0 이하 값 없음 -> 별도 처리 불필요")


[확인] 외국인 데이터에 결측/0 이하 값 없음 -> 별도 처리 불필요


## 2-3. AGE_CD 한글 라벨 매핑 확인

In [4]:
print(AGE_LABELS)
foreign_df[['AGE_CD', 'AGE_LABEL']].drop_duplicates().sort_values('AGE_CD')


{'1': '20대 이하', '2': '20대', '3': '30대', '4': '40대', '5': '50대', '6': '60대 이상'}


,AGE_CD,AGE_LABEL
4,1,20대 이하
3,2,20대
6,3,30대
1,4,40대
0,5,50대
11,6,60대 이상


## 2-4. 기초 통계 (§3-3 사전탐색 수치와 대조)

요구사항 §3-3에 제시된 예시치(약 1조 2,781억원 / 약 7,550만 건, 전체의 7.4~8.0%)와
전수 재계산 결과를 대조한다.

In [5]:
total_amt = foreign_df['amt'].sum()
total_cnt = foreign_df['cnt'].sum()
share_amt = total_amt / raw['amt'].sum() * 100
share_cnt = total_cnt / raw['cnt'].sum() * 100

print(f"외국인 총 소비금액: {total_amt:,}원 (약 {total_amt/1e8:,.0f}억원)")
print(f"외국인 총 이용건수: {total_cnt:,}건 (약 {total_cnt/1e4:,.0f}만 건)")
print(f"전체 대비 금액 비중: {share_amt:.1f}%  / 전체 대비 건수 비중: {share_cnt:.1f}%")


외국인 총 소비금액: 1,278,089,570,000원 (약 12,781억원)
외국인 총 이용건수: 75,509,726건 (약 7,551만 건)
전체 대비 금액 비중: 7.4%  / 전체 대비 건수 비중: 8.0%


In [6]:
monthly = foreign_df.groupby('STRD_YYMM')['amt'].sum()
print("월별 소비금액 추이:")
print(monthly.apply(lambda x: f"{x/1e8:,.0f}억원"))
print(f"\n최고점 월: {monthly.idxmax()} ({monthly.max()/1e8:,.0f}억원)")


월별 소비금액 추이:
STRD_YYMM
202601    2,073억원
202602    1,996억원
202603    2,128억원
202604    2,106억원
202605    2,337억원
202606    2,141억원
Name: amt, dtype: object

최고점 월: 202605 (2,337억원)


In [7]:
age_amt = foreign_df.groupby('AGE_LABEL')['amt'].sum().sort_values(ascending=False)
print("연령대별 소비금액 순위:")
print(age_amt.apply(lambda x: f"{x/1e8:,.0f}억원"))


연령대별 소비금액 순위:
AGE_LABEL
30대       3,070억원
40대       2,693억원
50대       2,465억원
20대       2,347억원
60대 이상    1,985억원
20대 이하      221억원
Name: amt, dtype: object


## 2-5. 저장

In [8]:
save_processed(foreign_df)
print(f"[OK] 저장 완료: {PROCESSED_PATH}")
print(f"[OK] 시군구(시도+시군구 복합키) 수: {foreign_df.groupby(['SIDO_NM','CCG_NM']).ngroups}")


[OK] 저장 완료: /home/hyunjinhwang/bc-card-data-contest/data/processed/foreign_consumption_clean.csv
[OK] 시군구(시도+시군구 복합키) 수: 255


## 요약

- 외국인 소비: 전수 재계산 결과와 §3-3 예시치가 거의 정확히 일치함 (약 1.28조원 / 약 7,551만 건, 전체의 약 7.5~7.9%).
- 5월이 월별 최고점이라는 예시치도 재확인됨.
- 결측/이상치 처리 로직 불필요 — 원본이 이미 clean한 집계 데이터.

다음 단계(`03_segmentation.ipynb`)에서 시군구 단위 특징 벡터를 만들고 3대 세그먼트로 분류한다.